# Extracting, Transforming and Loading Data (ETL)

In [1]:
# Import libraries
import pandas as pd
from pathlib import Path

# List of years for which we want to load survey data.
YEARS = [2022, 2023, 2024]

# Resolve project root so this notebook works whether CWD is repo root or notebooks/.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data" / "raw").exists()), Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
print(f"Working directory: {Path.cwd()}")
print(f"Using RAW_DIR: {RAW_DIR}")

# Initialize an empty list to hold DataFrames for each year.
frames = []

Working directory: c:\Users\aliff\Desktop\Local Repo\salary-prediction\notebooks
Using RAW_DIR: c:\Users\aliff\Desktop\Local Repo\salary-prediction\data\raw


In [2]:
# Cell 0 — notebook settings
# Load the autoreload extension to automatically reload modules before executing code
%load_ext autoreload 
# Set autoreload mode to 2, which means all modules will be reloaded before executing code
%autoreload 2 

# Suppress warnings to keep the notebook output clean, especially during development when there may be deprecation warnings or other non-critical warnings that can be safely ignored.
import warnings
warnings.filterwarnings("ignore")
# Set the maximum number of columns to display when printing DataFrames to 50
pd.set_option("display.max_columns", 50) 
# Set the float format for displaying DataFrames to two decimal places
pd.set_option("display.float_format", "{:,.0f}".format) 


# Load
### combine all years' data into a single DataFrame with a new 'survey_year' column to track the year of each row.

In [3]:
# Iterate through each year, read the corresponding CSV file, and append to the list.
for year in YEARS:
    path = RAW_DIR / str(year) / "survey_results_public.csv"
    
    # Check if file path exists. If not, raise FileNotFoundError
    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}. Run scripts/acquire.sh first to download/extract data."
        )

    # Read the CSV file into a DataFrame
    df = pd.read_csv(path, low_memory=False)
    df["survey_year"] = year # Add new column 'year' set to its year
    frames.append(df)        # Append the DataFrame to the list of frames for later concatenation.
    
    # Print the number of rows and columns for each year's DataFrame 
    print(f"{year}: {len(df):,} rows, {df.shape[1]} cols") 

2022: 73,268 rows, 80 cols
2023: 89,184 rows, 85 cols
2024: 65,437 rows, 115 cols


In [4]:
# After processing all years, concatenate the list of DataFrames into a single DataFrame. 
# The ignore_index=True option is used to reset the index in the combined DataFrame, 
# ensuring that it runs sequentially from 0 to n-1, 
# which is important for data integrity.
df_raw = pd.concat(frames, ignore_index=True)

## Inspect Extracted Data

In [5]:
# Check the total number of rows and columns in the combined DataFrame
print(f"\nCombined: {len(df_raw):,} rows, {df_raw.shape[1]} cols")


Combined: 227,889 rows, 144 cols


In [6]:
# Check the column names of the combined DataFrame
print("\nColumn names")
print(df_raw.columns.tolist())


Column names
['ResponseId', 'MainBranch', 'Employment', 'RemoteWork', 'CodingActivities', 'EdLevel', 'LearnCode', 'LearnCodeOnline', 'LearnCodeCoursesCert', 'YearsCode', 'YearsCodePro', 'DevType', 'OrgSize', 'PurchaseInfluence', 'BuyNewTool', 'Country', 'Currency', 'CompTotal', 'CompFreq', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'OpSysProfessional use', 'OpSysPersonal use', 'VersionControlSystem', 'VCInteraction', 'VCHostingPersonal use', 'VCHostingProfessional use', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OfficeStackSyncHaveWorkedWith', 'OfficeStackSyncWantToWorkWith', 'Blockchain', 'NEWSOSites', 'SOVisitFreq', 'SOAccount',

In [7]:
# Check the value counts of data types in the combined DataFrame to get an overview of the distribution of different data types, 
# which can help identify if there are any unexpected data types that may require attention during data cleaning or preprocessing.
print("\ndf_raw.dtypes value counts")
print(df_raw.dtypes.value_counts())

# Check the data types of each column in the combined DataFrame to understand the structure of the dataset and 
# identify any potential issues with data types that may need to be addressed during data cleaning or preprocessing.
print("\ndf_raw.dtypes")
print(df_raw.dtypes)



df_raw.dtypes value counts
str        127
float64     15
int64        2
Name: count, dtype: int64

df_raw.dtypes
ResponseId            int64
MainBranch              str
Employment              str
RemoteWork              str
CodingActivities        str
                     ...   
JobSatPoints_8      float64
JobSatPoints_9      float64
JobSatPoints_10     float64
JobSatPoints_11     float64
JobSat              float64
Length: 144, dtype: object


In [8]:
# Check the first few rows of the combined DataFrame to verify 
# that the data has been loaded and combined correctly, 
print(df_raw.head()) 

   ResponseId                                         MainBranch  \
0           1                                      None of these   
1           2                     I am a developer by profession   
2           3  I am not primarily a developer, but I write co...   
3           4                     I am a developer by profession   
4           5                     I am a developer by profession   

            Employment                            RemoteWork  \
0                  NaN                                   NaN   
1  Employed, full-time                          Fully remote   
2  Employed, full-time  Hybrid (some remote, some in-person)   
3  Employed, full-time                          Fully remote   
4  Employed, full-time  Hybrid (some remote, some in-person)   

                           CodingActivities  \
0                                       NaN   
1  Hobby;Contribute to open-source projects   
2                                     Hobby   
3              I d

In [9]:
# Check if the 'ConvertedCompYearly' column exists in the combined DataFrame. 
# If it does, print summary statistics and counts of non-null and null values to understand the distribution of salary data in the dataset.
print(df_raw["ConvertedCompYearly"].describe())
print(f"Non-null salary rows: {df_raw['ConvertedCompYearly'].notna().sum():,}")
print(f"Null salary rows:     {df_raw['ConvertedCompYearly'].isna().sum():,}")


count      109,525
mean       122,998
std        651,566
min              1
25%         38,556
50%         70,380
75%        120,000
max     74,351,432
Name: ConvertedCompYearly, dtype: float64
Non-null salary rows: 109,525
Null salary rows:     118,364


In [10]:
# Check the distribution of non-null salary values by printing summary statistics including specific percentiles.
salary = df_raw["ConvertedCompYearly"].dropna()
print(salary.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

count      109,525
mean       122,998
std        651,566
min              1
1%             469
5%           5,988
25%         38,556
50%         70,380
75%        120,000
95%        250,000
99%        904,608
max     74,351,432
Name: ConvertedCompYearly, dtype: float64


In [11]:
# Inspect nulls by calculating the percentage of null values in each column, sorting them in descending order, and displaying the top 30 columns with null values. 
# This helps identify which features have the most missing data and may require attention during data cleaning or feature engineering.
null_pct = df_raw.isnull().mean().sort_values(ascending=False)
null_pct[null_pct > 0.0].head(30)

VCHostingProfessional use             1
VCHostingPersonal use                 1
AINextMuch less integrated            1
AINextLess integrated                 1
AINextVery similar                    1
AINextSomewhat similar                1
AINextNeither different nor similar   1
AINextNo change                       1
AINextVery different                  1
AINextMuch more integrated            1
EmbeddedAdmired                       1
EmbeddedWantToWorkWith                1
AIDevWantToWorkWith                   1
EmbeddedHaveWorkedWith                1
AINextSomewhat different              1
AINextMore integrated                 1
AIDevHaveWorkedWith                   1
Knowledge_9                           1
Frustration                           1
ProfessionalCloud                     1
ProfessionalQuestion                  1
JobSat                                1
JobSatPoints_1                        1
JobSatPoints_4                        1
JobSatPoints_5                        1


# Cleaning 

### Drop Null Salary Rows

In [12]:
# Drop rows with no salary (can't train without a target)
print(f"Before dropping null salary, have {len(df_raw):,} rows")
df = df_raw.dropna(subset=["ConvertedCompYearly"]).copy()
print(f"After dropping null salary, left with {len(df):,} rows")

Before dropping null salary, have 227,889 rows
After dropping null salary, left with 109,525 rows


### Remove Outliers Salaries

In [13]:
# Remove outliers by filtering the DataFrame to include only rows where 'ConvertedCompYearly' is between 10,000 and 400,000.
print(f"Before outlier removal: {len(df):,} rows")
df = df[df["ConvertedCompYearly"].between(10_000, 400_000)]
print(f"After outlier removal left with {len(df):,} rows")

Before outlier removal: 109,525 rows
After outlier removal left with 98,779 rows


### YearsCodePro to Numeric

In [ ]:
# Convert YearsCodePro to numeric robustly (works for mapped labels and numeric-like strings)
YCP_MAP = {
    "Less than 1 year": 0.5,
    "More than 50 years": 52.0,
}

# Keep known labels from map; parse everything else by extracting digits
ycp_raw = df["YearsCodePro"]
ycp_mapped = ycp_raw.map(YCP_MAP)
ycp_parsed = pd.to_numeric(
    ycp_raw.astype("string").str.extract(r"(\d+)", expand=False),
    errors="coerce",
)

df["YearsCodePro"] = ycp_mapped.fillna(ycp_parsed).astype(float)

Non-numeric count: 0
Series([], Name: count, dtype: int64)
Empty DataFrame
Columns: [ResponseId, MainBranch, CodingActivities, LearnCode, LearnCodeOnline, LearnCodeCoursesCert, YearsCode, YearsCodePro, DevType, OrgSize, PurchaseInfluence, BuyNewTool, Country, Currency, CompTotal, CompFreq, LanguageHaveWorkedWith, LanguageWantToWorkWith, DatabaseHaveWorkedWith, DatabaseWantToWorkWith, PlatformHaveWorkedWith, PlatformWantToWorkWith, WebframeHaveWorkedWith, WebframeWantToWorkWith, MiscTechHaveWorkedWith, MiscTechWantToWorkWith, ToolsTechHaveWorkedWith, ToolsTechWantToWorkWith, NEWCollabToolsHaveWorkedWith, NEWCollabToolsWantToWorkWith, OpSysProfessional use, OpSysPersonal use, VersionControlSystem, VCInteraction, VCHostingPersonal use, VCHostingProfessional use, OfficeStackAsyncHaveWorkedWith, OfficeStackAsyncWantToWorkWith, OfficeStackSyncHaveWorkedWith, OfficeStackSyncWantToWorkWith, Blockchain, NEWSOSites, SOVisitFreq, SOAccount, SOPartFreq, SOComm, Age, Gender, Trans, Sexuality, Ethni

### Binary Encode Top 20 Languages

In [14]:
# Encode multi-select columns 
# # The "LanguageHaveWorkedWith" column contains multiple programming languages separated by semicolons.
# The str.get_dummies() method is used to create a binary indicator (one-hot encoding)
# for each of the top 20 most common programming languages, which can then be used as features in machine learning models.
print(f"Before encoding languages, have {df['LanguageHaveWorkedWith'].nunique()} unique language combinations")
lang_dummies = df["LanguageHaveWorkedWith"].str.get_dummies(sep=";")
top20_langs  = lang_dummies.sum().nlargest(20).index
df = pd.concat([df, lang_dummies[top20_langs].add_prefix("lang_")], axis=1)
print(f"After encoding languages, have {len(top20_langs)} language features: {top20_langs.tolist()}")

Before encoding languages, have 33718 unique language combinations
After encoding languages, have 20 language features: ['JavaScript', 'SQL', 'HTML/CSS', 'Python', 'TypeScript', 'C#', 'Java', 'Bash/Shell (all shells)', 'PHP', 'C++', 'PowerShell', 'Go', 'C', 'Bash/Shell', 'Rust', 'Kotlin', 'Ruby', 'Swift', 'Dart', 'Lua']


### One Hot Encode RemoteWork, EdLevel, Employment 

In [15]:
# One-hot encode single-select categoricals (idempotent: only encode columns that still exist)
cols_to_encode = [c for c in ["RemoteWork", "EdLevel", "Employment"] if c in df.columns]
if cols_to_encode:
    df = pd.get_dummies(df, columns=cols_to_encode, dummy_na=False)
    print(f"Encoded: {cols_to_encode}")
else:
    print("Already encoded — skipping.")

Encoded: ['RemoteWork', 'EdLevel', 'Employment']


### Standardise Country 

In [16]:
# Standardise Country 
# Group countries with fewer than 100 respondents into "Other" to reduce the number of unique categories and help prevent overfitting in models that use the 'Country' feature.
print(f"Before grouping rare countries, have {df['Country'].nunique()} unique countries")
country_counts = df["Country"].value_counts()
rare_countries = country_counts[country_counts < 100].index
df["Country"] = df["Country"].replace(rare_countries, "Other")
print(f"After grouping rare countries, left with {df['Country'].nunique()} unique countries")

Before grouping rare countries, have 174 unique countries
After grouping rare countries, left with 71 unique countries


In [17]:
# Save
out = Path(PROJECT_ROOT / "data" / "clean")
out.mkdir(exist_ok=True)
df.to_csv(out / "survey_clean.csv", index=False)
print(f"\nSaved {len(df):,} rows, {df.shape[1]} cols → {out / 'survey_clean.csv'}")

PermissionError: [Errno 13] Permission denied: 'c:\\Users\\aliff\\Desktop\\Local Repo\\salary-prediction\\data\\clean\\survey_clean.csv'